# Displacement

**Project:** Pali Wacth

**Notebook:** `displacement.ipynb`

**Author:** Yuseof J

**Created:** 18/11/2026 | **Last updated:** 18/11/2026

## Purpose
   This file analyzes displacement pressures by governorate

## Inputs

- `data/raw/boundaries/`: HDX admin boundaries (COD-AB)

## Outputs
- `data/processed/[filename].csv`: what it contains

## Notes and caveats
    - Data runs from Jan 2025 to Oct 2026, so historical analysis is limited accordingly

In [ ]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

CRS = 'EPSG:4326'

### Data imports / exports

In [ ]:
# import paths
PATH_TO_DISPLACEMENT_EVENTS = '../data/raw/pse_idmc_idu_events.csv'
PATH_TO_ADMIN_BOUNDARIES = '../data/raw/pse_admin_boundaries.shp/pse_admin2.shp'

# export paths
PATH_TO_GEOLOCATED_EVENTS_GPKG = '../data/processed/acled/conflict_events_geolocated.gpkg'
PATH_TO_GEOLOCATED_EVENTS_DF = '../data/processed/acled/conflict_events_geolocated.csv'

# import data
df_displacement = pd.read_csv(PATH_TO_DISPLACEMENT_EVENTS)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

# EDA

In [ ]:
# df_displacement.columns.tolist()

In [ ]:
df_displacement.head()

## Geographic Distribution

In [ ]:
print(f"{df_displacement.centroid.nunique()} unique centroids")
print(f"{df_displacement.latitude.nunique()} unique latitudes")
print(f"{df_displacement.longitude.nunique()} unique longitudes")
print(f"{df_displacement.locations_type.nunique()} unique locations_type")
print(f"{df_displacement.locations_name.nunique()} unique locations_name")

### TODO: rich info in locations_name, can be distilled for finer geographic granularity

In [ ]:
# df_displacement.locations_name.tolist()

## Temporal Distribution

### TODO: date columns don't always align with 'description' column (which contains dates of displacement

In [ ]:
print(f"Earliest start date: {df_displacement.displacement_start_date.min()}")
print(f"Latest start date: {df_displacement.displacement_start_date.max()}")
print(f"Earliest end date: {df_displacement.displacement_end_date.min()}")
print(f"Latest end date: {df_displacement.displacement_end_date.max()}")

In [ ]:
#TODO data columns

# Geoprocessing

In [ ]:
# convert event coordinates into spatial points
gdf_displacement = gpd.GeoDataFrame(
    df_displacement,
    geometry=gpd.points_from_xy(
        df_displacement['longitude'],
        df_displacement['latitude']
    ),
    crs=CRS
)

In [ ]:
# check and align CRSs
print("Events CRS:", gdf_displacement.crs)
print("Admin CRS:", gdf_boundaries.crs)
gdf_displacement = gdf_displacement.to_crs(gdf_boundaries.crs) # failsafe -- convert events CRS to admin boundaries CRS if not already aligned
assert(gdf_displacement.crs == gdf_boundaries.crs)
print('Confirmed CRSs are aligned')

In [ ]:
# perform spatial join (assign event coordinates to an administrative boundary)
gdf_displacement_joined = gpd.sjoin(
    gdf_displacement,
    gdf_boundaries[['adm1_name', 'adm1_pcode', 'adm2_name', 'adm2_pcode', 'geometry']],
    how='left',
    predicate='within'
)

In [ ]:
print("Total events:", len(gdf_displacement_joined))
print("Matched events:", gdf_displacement_joined['adm1_name'].notna().sum())
print("Unmatched events:", gdf_displacement_joined['adm1_name'].isna().sum())

### TODO: handle unmatched events

In [ ]:
unmatched = gdf_events_joined[gdf_events_joined['adm1_name'].isna()]
if len(unmatched) > 0:
    display(unmatched.head())
else:
    print('All events matched successfully!')

# FINDING: Where is displacement happening (overall/across time)?

**While the West Bank has seen a far greater NUMBER of displacement events, Gaza experienced a far greater AMOUNT of displaced individuals**

Num displacement events per admin 

In [ ]:
 gdf_displacement_joined['adm1_name'].value_counts()

In [ ]:
 gdf_displacement_joined.groupby(['adm1_name']).adm2_name.value_counts()

Num displaced people per admin

In [ ]:
gdf_displacement_joined.groupby('adm1_name').figure.sum()

In [ ]:
gdf_displacement_joined.groupby(['adm1_name', 'adm2_name']).figure.sum()